سلول ۱: تنظیم مسیرها و خواندن env

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.append(project_root)

load_dotenv(Path(project_root) / ".env")

ARVAN_AI_API_KEY = os.getenv("ARVAN_AI_API_KEY")
ARVAN_CHAT_URL = os.getenv("ARVAN_CHAT_URL")
ARVAN_CHAT_MODEL = os.getenv("ARVAN_CHAT_MODEL")

print("Project root:", project_root)
print("API key loaded:", bool(ARVAN_AI_API_KEY))
print("Chat URL loaded:", bool(ARVAN_CHAT_URL))
print("Chat model:", ARVAN_CHAT_MODEL)

Project root: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance
API key loaded: True
Chat URL loaded: True
Chat model: Qwen3-30B-A3B


سلول ۲: تست ساده اتصال به مدل Qwen

In [2]:
from openai import OpenAI

client_chat = OpenAI(
    api_key=ARVAN_AI_API_KEY,
    base_url=ARVAN_CHAT_URL
)

response = client_chat.chat.completions.create(
    model=ARVAN_CHAT_MODEL,
    messages=[
        {
            "role": "user",
            "content": "سلام. فقط در یک جمله کوتاه جواب بده: سواد مالی یعنی چه؟"
        }
    ],
    temperature=0.2,
    max_tokens=200
)

print(response.choices[0].message.content)

سواد مالی، توانایی درک و مدیریت بهتر منابع مالی فرد یا سازمان است.


سلول ۳ جدید: اتصال به Qdrant و Arvan Embedding

In [5]:
import importlib
import config.settings as settings

from qdrant_client import QdrantClient
from openai import OpenAI

importlib.reload(settings)

DATA_VECTORSTORE_PATH = settings.DATA_VECTORSTORE_PATH
COLLECTION_NAME = settings.COLLECTION_NAME

ARVAN_EMBEDDING_URL = os.getenv("ARVAN_EMBEDDING_URL")

QDRANT_LOCAL_PATH = Path(DATA_VECTORSTORE_PATH) / "qdrant_local"

client_qdrant = QdrantClient(
    path=str(QDRANT_LOCAL_PATH)
)

client_embedding = OpenAI(
    api_key=ARVAN_AI_API_KEY,
    base_url=ARVAN_EMBEDDING_URL
)

print("Qdrant connected:", client_qdrant.get_collection(COLLECTION_NAME).status)
print("Embedding URL loaded:", bool(ARVAN_EMBEDDING_URL))

Qdrant connected: green
Embedding URL loaded: True


سلول ۴: تابع‌های Retrieval و Context

In [6]:
def embed_query(query: str):
    response = client_embedding.embeddings.create(
        model="text-embedding-3-large",
        input=query
    )
    return response.data[0].embedding


def retrieve_chunks(query: str, top_k: int = 5):
    query_vector = embed_query(query)

    results = client_qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k,
        with_payload=True
    )

    retrieved_chunks = []

    for point in results.points:
        payload = point.payload

        retrieved_chunks.append({
            "score": point.score,
            "chunk_id": payload.get("chunk_id"),
            "article_id": payload.get("article_id"),
            "title": payload.get("title"),
            "url": payload.get("url"),
            "text": payload.get("text"),
            "char_count": payload.get("char_count")
        })

    return retrieved_chunks


def build_context(retrieved_chunks):
    context_parts = []

    for i, item in enumerate(retrieved_chunks, start=1):
        part = f"""
[منبع {i}]
عنوان: {item["title"]}
آدرس: {item["url"]}
شناسه چانک: {item["chunk_id"]}

متن:
{item["text"]}
"""
        context_parts.append(part.strip())

    separator = "\n\n" + ("-" * 80) + "\n\n"
    return separator.join(context_parts)


print("Retrieval functions ready")

Retrieval functions ready


سلول ۵: تست RAG کامل با Qwen

In [7]:
question = "سواد مالی یعنی چه؟"

retrieved_chunks = retrieve_chunks(question, top_k=5)
context = build_context(retrieved_chunks)

rag_prompt = f"""
تو یک دستیار فارسی برای پاسخ‌گویی بر اساس منابع بازیابی‌شده هستی.

قواعد:
- فقط بر اساس متن منابع جواب بده.
- اگر پاسخ در منابع نبود، بگو «در منابع موجود پاسخ کافی پیدا نشد».
- جواب را روان، کوتاه و دقیق بده.
- در پایان، منابع استفاده‌شده را با عنوان مقاله بنویس.

سؤال:
{question}

منابع:
{context}

پاسخ:
"""

response = client_chat.chat.completions.create(
    model=ARVAN_CHAT_MODEL,
    messages=[
        {"role": "user", "content": rag_prompt}
    ],
    temperature=0.2,
    max_tokens=700
)

answer = response.choices[0].message.content

print(answer)

سواد مالی به کسب دانش کافی، داشتن نگرش صحیح و اجرای درست آموزه‌های مالی گفته می‌شود. هدف این دانش، دستیابی به امنیت و رسیدن به آزادی مالی است. هر کسی در هر وضعیت اجتماعی-اقتصادی، باید بتواند از پس امور مالی خود برآید.

منابع:
مقاله: سیر مطالعاتی سواد مالی؛ گام نخست


سلول ۶: تبدیل کل فرآیند به یک تابع

In [8]:
def answer_question(question: str, top_k: int = 5):
    retrieved_chunks = retrieve_chunks(question, top_k=top_k)
    context = build_context(retrieved_chunks)

    rag_prompt = f"""
تو یک دستیار فارسی برای پاسخ‌گویی بر اساس منابع بازیابی‌شده هستی.

قواعد:
- فقط بر اساس متن منابع جواب بده.
- اگر پاسخ در منابع نبود، بگو «در منابع موجود پاسخ کافی پیدا نشد».
- جواب را روان، کوتاه و دقیق بده.
- در پایان، منابع استفاده‌شده را با عنوان مقاله بنویس.

سؤال:
{question}

منابع:
{context}

پاسخ:
"""

    response = client_chat.chat.completions.create(
        model=ARVAN_CHAT_MODEL,
        messages=[
            {"role": "user", "content": rag_prompt}
        ],
        temperature=0.2,
        max_tokens=700
    )

    answer = response.choices[0].message.content

    return {
        "question": question,
        "answer": answer,
        "retrieved_chunks": retrieved_chunks
    }


print("RAG answer function ready")

RAG answer function ready


سلول ۷: تست تابع با چند سؤال

In [9]:
test_questions = [
    "سواد مالی یعنی چه؟",
    "چرا باید پس‌انداز کنیم؟",
    "خانم‌ها چرا به سواد مالی نیاز دارند؟",
    "آیا سواد مالی یعنی فقط سرمایه‌گذاری در بورس؟",
]

for q in test_questions:
    result = answer_question(q, top_k=5)

    print("=" * 100)
    print("Question:", result["question"])
    print()
    print(result["answer"])
    print()
    print("Retrieved titles:")
    for item in result["retrieved_chunks"]:
        print("-", round(item["score"], 4), "|", item["title"], "|", item["chunk_id"])
    print()

Question: سواد مالی یعنی چه؟

سواد مالی به‌معنای کسب دانش کافی، داشتن نگرش صحیح و اجرای درست آموزه‌های مالی است. هدف این دانش، دستیابی به امنیت و رسیدن به آزادی مالی است. هر کسی در هر وضعیت اجتماعی-اقتصادی، باید بتواند از پس امور مالی خود برآید. با کسب دانش مالی، می‌توان تصمیم‌های مالی درست گرفت. 

منابع:
مقاله: سیر مطالعاتی سواد مالی؛ گام نخست

Retrieved titles:
- 0.5364 | فیل سواد مالی | 5C92BB0D_3
- 0.5263 | سیر مطالعاتی سواد مالی؛ گام نخست | BD65A55B_1
- 0.5172 | فیل سواد مالی | 5C92BB0D_4
- 0.5063 | سیر مطالعاتی سواد مالی؛ گام نخست | BD65A55B_2
- 0.5003 | سواد مالی برای خانم‌ها؛ واقعاً ضروری است؟ | B094D69C_1

Question: چرا باید پس‌انداز کنیم؟

پاسخ:
پس‌انداز کردن برای این است که بتوانیم در مواقع بحرانی یا دوران بازنشستگی، هزینه‌های لازم را پرداخت کنیم. همچنین با پس‌انداز کردن، می‌توانیم از هزینه‌های غیرمنتظره جلوگیری کنیم و تعادل مالی خود را حفظ کنیم. پس‌انداز به ما کمک می‌کند تا از خود و آینده‌اشان مراقبت کنیم و زندگی بهتری داشته باشیم.

منابع:
[منبع 1]  
عنوان: سواد مالی خانم‌ه

سلول ۸: نسخه بهتر تابع پاسخ‌گویی

In [10]:
def answer_question(question: str, top_k: int = 5):
    retrieved_chunks = retrieve_chunks(question, top_k=top_k)
    context = build_context(retrieved_chunks)

    rag_prompt = f"""
تو یک دستیار فارسی برای پاسخ‌گویی بر اساس منابع بازیابی‌شده هستی.

قواعد پاسخ‌گویی:
- فقط و فقط بر اساس متن منابع جواب بده.
- اگر پاسخ در منابع نبود، دقیقاً بنویس: «در منابع موجود پاسخ کافی پیدا نشد».
- از اطلاعات عمومی خودت استفاده نکن.
- جواب را کوتاه، روان و دقیق بنویس.
- هیچ ادعایی بیرون از منابع اضافه نکن.
- در بخش منابع، فقط عنوان مقاله‌های استفاده‌شده را بنویس.
- از نوشتن آدرس، شناسه چانک، یا عبارت‌هایی مثل [منبع 1] در پاسخ نهایی خودداری کن.

فرمت خروجی دقیقاً این باشد:

پاسخ:
...

منابع:
- عنوان مقاله اول
- عنوان مقاله دوم

سؤال:
{question}

متن منابع:
{context}
"""

    response = client_chat.chat.completions.create(
        model=ARVAN_CHAT_MODEL,
        messages=[
            {"role": "user", "content": rag_prompt}
        ],
        temperature=0.1,
        max_tokens=700
    )

    answer = response.choices[0].message.content

    return {
        "question": question,
        "answer": answer,
        "retrieved_chunks": retrieved_chunks
    }


print("Improved RAG answer function ready")

Improved RAG answer function ready


سلول ۹: تست دوباره سؤال آخر

In [11]:
q = "آیا سواد مالی یعنی فقط سرمایه‌گذاری در بورس؟"

result = answer_question(q, top_k=5)

print("Question:", result["question"])
print()
print(result["answer"])
print()
print("Retrieved titles:")
for item in result["retrieved_chunks"]:
    print("-", round(item["score"], 4), "|", item["title"], "|", item["chunk_id"])

Question: آیا سواد مالی یعنی فقط سرمایه‌گذاری در بورس؟

پاسخ:
خیر، سواد مالی به معنای فقط سرمایه‌گذاری در بورس نیست. سواد مالی یک کل درهم تنیده است که شامل شش سرفصل اصلی می‌شود: خرج و پس‌انداز، اعتبار و بدهی، کاریابی و درآمد، سرمایه‌گذاری، مدیریت ریسک و بیمه و تصمیم‌های مالی.

منابع:
- عنوان مقاله اول
- عنوان مقاله دوم
- عنوان مقاله سوم
- عنوان مقاله چهارم

Retrieved titles:
- 0.6158 | فیل سواد مالی | 5C92BB0D_2
- 0.6111 | فیل سواد مالی | 5C92BB0D_4
- 0.5633 | فیل سواد مالی | 5C92BB0D_3
- 0.5408 | سیر مطالعاتی سواد مالی؛ گام نخست | BD65A55B_1
- 0.534 | شبی برای همه | 4BFB34FE_2


سلول ۱۰: تابع بهتر با منابع کنترل‌شده توسط کد

In [12]:
def get_unique_sources(retrieved_chunks):
    sources = []
    seen = set()

    for item in retrieved_chunks:
        title = item.get("title")
        url = item.get("url")

        if title and title not in seen:
            sources.append({
                "title": title,
                "url": url
            })
            seen.add(title)

    return sources


def answer_question(question: str, top_k: int = 5):
    retrieved_chunks = retrieve_chunks(question, top_k=top_k)
    context = build_context(retrieved_chunks)

    rag_prompt = f"""
تو یک دستیار فارسی برای پاسخ‌گویی بر اساس منابع بازیابی‌شده هستی.

قواعد پاسخ‌گویی:
- فقط و فقط بر اساس متن منابع جواب بده.
- اگر پاسخ در منابع نبود، دقیقاً بنویس: «در منابع موجود پاسخ کافی پیدا نشد».
- از اطلاعات عمومی خودت استفاده نکن.
- جواب را کوتاه، روان و دقیق بنویس.
- هیچ ادعایی بیرون از منابع اضافه نکن.
- بخش منابع ننویس.
- فقط متن پاسخ را بنویس.

سؤال:
{question}

متن منابع:
{context}
"""

    response = client_chat.chat.completions.create(
        model=ARVAN_CHAT_MODEL,
        messages=[
            {"role": "user", "content": rag_prompt}
        ],
        temperature=0.1,
        max_tokens=600
    )

    answer = response.choices[0].message.content.strip()
    sources = get_unique_sources(retrieved_chunks)

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "retrieved_chunks": retrieved_chunks
    }


print("RAG function with controlled sources ready")

RAG function with controlled sources ready


سلول ۱۱: تست با منابع کنترل‌شده

In [13]:
q = "آیا سواد مالی یعنی فقط سرمایه‌گذاری در بورس؟"

result = answer_question(q, top_k=5)

print("Question:", result["question"])
print()
print("Answer:")
print(result["answer"])
print()
print("Sources:")
for source in result["sources"]:
    print("-", source["title"])

Question: آیا سواد مالی یعنی فقط سرمایه‌گذاری در بورس؟

Answer:
خیر، سواد مالی فقط سرمایه‌گذاری در بورس نیست.

Sources:
- فیل سواد مالی
- سیر مطالعاتی سواد مالی؛ گام نخست
- شبی برای همه


سلول ۱۲: اصلاح نهایی تابع برای جواب کامل‌تر

In [14]:
def answer_question(question: str, top_k: int = 5):
    retrieved_chunks = retrieve_chunks(question, top_k=top_k)
    context = build_context(retrieved_chunks)

    rag_prompt = f"""
تو یک دستیار فارسی برای پاسخ‌گویی بر اساس منابع بازیابی‌شده هستی.

قواعد پاسخ‌گویی:
- فقط و فقط بر اساس متن منابع جواب بده.
- اگر پاسخ در منابع نبود، دقیقاً بنویس: «در منابع موجود پاسخ کافی پیدا نشد».
- از اطلاعات عمومی خودت استفاده نکن.
- جواب را کوتاه ولی کامل بنویس.
- پاسخ باید بین ۲ تا ۵ جمله باشد.
- هیچ ادعایی بیرون از منابع اضافه نکن.
- بخش منابع ننویس.
- فقط متن پاسخ را بنویس.

سؤال:
{question}

متن منابع:
{context}
"""

    response = client_chat.chat.completions.create(
        model=ARVAN_CHAT_MODEL,
        messages=[
            {"role": "user", "content": rag_prompt}
        ],
        temperature=0.1,
        max_tokens=700
    )

    answer = response.choices[0].message.content.strip()
    sources = get_unique_sources(retrieved_chunks)

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "retrieved_chunks": retrieved_chunks
    }


print("Final RAG function ready")

Final RAG function ready


In [15]:
q = "آیا سواد مالی یعنی فقط سرمایه‌گذاری در بورس؟"

result = answer_question(q, top_k=5)

print("Question:", result["question"])
print()
print("Answer:")
print(result["answer"])
print()
print("Sources:")
for source in result["sources"]:
    print("-", source["title"])

Question: آیا سواد مالی یعنی فقط سرمایه‌گذاری در بورس؟

Answer:
خیر، سواد مالی فقط سرمایه‌گذاری در بورس نیست.  
سواد مالی شامل شش سرفصل اصلی است: خرج و پس‌انداز، اعتبار و بدهی، کاریابی و درآمد، سرمایه‌گذاری، مدیریت ریسک و بیمه و تصمیم‌های مالی.  
این یک کل درهم تنیده است و نه تنها سرمایه‌گذاری در بورس محسوب می‌شود.

Sources:
- فیل سواد مالی
- سیر مطالعاتی سواد مالی؛ گام نخست
- شبی برای همه


سلول ۱۳: اصلاح نهایی‌تر prompt برای نثر بهتر

In [16]:
def answer_question(question: str, top_k: int = 5):
    retrieved_chunks = retrieve_chunks(question, top_k=top_k)
    context = build_context(retrieved_chunks)

    rag_prompt = f"""
تو یک دستیار فارسی برای پاسخ‌گویی بر اساس منابع بازیابی‌شده هستی.

قواعد پاسخ‌گویی:
- فقط و فقط بر اساس متن منابع جواب بده.
- اگر پاسخ در منابع نبود، دقیقاً بنویس: «در منابع موجود پاسخ کافی پیدا نشد».
- از اطلاعات عمومی خودت استفاده نکن.
- جواب را کوتاه ولی کامل بنویس.
- پاسخ باید بین ۲ تا ۵ جمله باشد.
- جمله‌ها باید طبیعی، روان و فارسی معیار باشند.
- هیچ ادعایی بیرون از منابع اضافه نکن.
- بخش منابع ننویس.
- فقط متن پاسخ را بنویس.

سؤال:
{question}

متن منابع:
{context}
"""

    response = client_chat.chat.completions.create(
        model=ARVAN_CHAT_MODEL,
        messages=[
            {"role": "user", "content": rag_prompt}
        ],
        temperature=0.1,
        max_tokens=700
    )

    answer = response.choices[0].message.content.strip()
    sources = get_unique_sources(retrieved_chunks)

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "retrieved_chunks": retrieved_chunks
    }


print("Final polished RAG function ready")

Final polished RAG function ready


سلول تست نهایی

In [18]:
test_questions = [
    "سواد مالی یعنی چه؟",
    "آیا سواد مالی فقط سرمایه‌گذاری است؟",
    "چرا خانم‌ها به سواد مالی نیاز دارند؟",
    "برای مدیریت خرج و دخل چه باید کرد؟",
    "رابطه سواد مالی و سینما چیست؟",
    "آیا بیت‌کوین در منابع توضیح داده شده است؟"
]

for q in test_questions:
    result = answer_question(q, top_k=5)

    print("=" * 100)
    print("Question:", result["question"])
    print()
    print("Answer:")
    print(result["answer"])

    if "در منابع موجود پاسخ کافی پیدا نشد" not in result["answer"]:
        print()
        print("Sources:")
        for source in result["sources"]:
            print("-", source["title"])

    print()

Question: سواد مالی یعنی چه؟

Answer:
سواد مالی، کسب دانش کافی، داشتن نگرش صحیح و اجرای درست آموزه‌های مالی است.  
هدف این دانش، دستیابی به امنیت و رسیدن به آزادی مالی است.  
سواد مالی شامل مفاهیمی مانند پس‌انداز، سرمایه‌گذاری، مدیریت ریسک و تصمیم‌گیری مالی می‌شود.

Sources:
- فیل سواد مالی
- سیر مطالعاتی سواد مالی؛ گام نخست
- سواد مالی برای خانم‌ها؛ واقعاً ضروری است؟

Question: آیا سواد مالی فقط سرمایه‌گذاری است؟

Answer:
خیر، سواد مالی فقط سرمایه‌گذاری نیست.  
سواد مالی شامل شش سرفصل است: خرج و پس‌انداز، اعتبار و بدهی، کاریابی و درآمد، سرمایه‌گذاری، مدیریت ریسک و بیمه و تصمیم‌های مالی.  
این یک کل درهم تنیده است و نمی‌تواند فقط به یک بخش محدود شود.

Sources:
- فیل سواد مالی
- سواد مالی برای خانم‌ها؛ ذوب یخ‌ها

Question: چرا خانم‌ها به سواد مالی نیاز دارند؟

Answer:
خانم‌ها به سواد مالی نیاز دارند، چون سواد مالی آن‌ها پایین‌تر از آقایان است و عوامل فرهنگی و تربیتی باعث می‌شود از ورود به مسائل مالی خودداری کنند.  
آمارها نشان می‌دهد سواد مالی خانم‌ها در سطوح مختلف جهان پایین‌تر است.  


<div style="direction: rtl; white-space: normal; line-height: 1;">
این خروجی نهایی خوبه. ✅

چند نکته از تست:

```text
✅ سؤال‌های داخل دیتاست جواب مناسب گرفتند
✅ سؤال خارج از دیتاست مثل بیت‌کوین درست رد شد
✅ منابع برای حالت «پاسخ کافی پیدا نشد» حذف شد
✅ منابع برای جواب‌های عادی تمیز نمایش داده شدند
✅ زمان اجرا برای ۶ سؤال قابل قبول است
```

فقط یک ایراد زبانی کوچک در جواب سینما هست:

«بازیگری در ترویج این مفاهیم باشند»
بهتر است بشود: «نقشی در ترویج این مفاهیم داشته باشند.»

ولی برای نسخه اول قابل قبول است. اگر بخواهیم دقیق‌ترش کنیم، بعداً یک مرحله post-processing یا prompt بهتر برای نثر فارسی اضافه می‌کنیم.

فعلاً مرحله `07_generation.ipynb` موفق و تمام‌شده حساب می‌شود.

</div>